### Importing Libraries

In [1]:
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

In [8]:
lenet_transforms = transforms.Compose([
    transforms.Resize((28, 28)),    
    transforms.ToTensor(),        
    transforms.Normalize((0.1307,), (0.3081,))
])

In [9]:
train_dataset = datasets.MNIST(
    root='./data', 
    train=True, 
    download=True, 
    transform=lenet_transforms
)

test_dataset = datasets.MNIST(
    root='./data', 
    train=False, 
    transform=lenet_transforms
)

100.0%
100.0%
100.0%
100.0%


In [18]:
train_loader = DataLoader(
    train_dataset,
    batch_size=5,
    shuffle=True,
)
test_loader = DataLoader(test_dataset, batch_size=5, shuffle=True)

#### Custom LeCun "SigmoidLike" Activation

$$
\begin{aligned}
x_{i} &= f(a_{i})\\
f(a) &= A \tanh (Sa)
\end{aligned}
$$

In [17]:
class SigmoidLike(nn.Module):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.A = 1.17159
        self.S = 2/3
        
    def forward(self, x):
        return self.A  * torch.tanh(self.S * x)

### Custom C3 Layer


In [19]:
class LeCunC3(nn.Module):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        C3_CONNECTION_MAP = [
            [0,1,2],
            [1,2,3],
            [2,3,4],
            [3,4,5],
            [0,4,5],
            [0,1,5],
            [0,1,2,3],
            [1,2,3,4],
            [2,3,4,5],
            [0,3,4,5],
            [0,1,4,5],
            [0,1,2,5],
            [0,1,3,4],
            [1,2,4,5],
            [0,2,3,5],
            [0,1,2,3,4,5]
        ]

        self.c3_featuremaps = nn.ModuleList([
            nn.Conv2d(in_channels=len(inputs),out_channels=1, kernel=5)
            for inputs in C3_CONNECTION_MAP
        ])

    def forward(self,x):
        outputs = []
        for conv, input_maps in zip(self.c3_featuremaps, C3_CONNECTION_MAP):
            x_subset = x[:, input_maps, :, :]
            outputs.append(conv(x_subset))

        return outputs



In [16]:
x = torch.arange(0,10)
y = torch.arange(1,10)


(tensor(0), tensor(1))
(tensor(1), tensor(2))
(tensor(2), tensor(3))
(tensor(3), tensor(4))
(tensor(4), tensor(5))
(tensor(5), tensor(6))
(tensor(6), tensor(7))
(tensor(7), tensor(8))
(tensor(8), tensor(9))


## Model

In [7]:
class LeNet5(nn.Module):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.conv1 = nn.Conv2d(in_channels=1,out_channels=6,kernel_size=5)
        self.avgpool1 = nn.AvgPool2d(kernel_size=2)
        self.conv2 = nn.Conv2d(in_channels=6,out_channels=16,kernel_size=5)
        self.avgpool2 = nn.AvgPool2d(kernel_size=2)
        self.conv3 = nn.Conv2d(in_channels=16,out_channels=120,kernel_size=4)
        self.fc1 = nn.Linear(in_features=120,out_features=84)

        self.act = SigmoidLike()


    
    def forward(self, x):
        x = self.conv1(x)
        x = self.act(x)

        x = self.avgpool1(x)
        x = self.act(x)

        x = self.conv2(x)
        x = self.act(x)

        x = self.avgpool2(x)
        x = self.act(x)

        x = self.conv3(x) 
        x = self.act(x)
        x = x.view(x.size(0), -1)

        x = self.fc1(x)

        return x

### Model Init

In [10]:
model = LeNet5()
model.to(device="cuda")

LeNet5(
  (conv1): Conv2d(1, 6, kernel_size=(5, 5), stride=(1, 1))
  (avgpool1): AvgPool2d(kernel_size=2, stride=2, padding=0)
  (conv2): Conv2d(6, 16, kernel_size=(5, 5), stride=(1, 1))
  (avgpool2): AvgPool2d(kernel_size=2, stride=2, padding=0)
  (conv3): Conv2d(16, 120, kernel_size=(4, 4), stride=(1, 1))
  (fc1): Linear(in_features=120, out_features=84, bias=True)
  (act): SigmoidLike()
)